## Limpeza

### Regras de limpeza:
* Post sem comentários serão removidos
* Nome de usuário do autor será trocado pelo id do link do post
* Nome de usuário dos comentários serão trocados por uid aleatórios por questões de privacidade
* Valor ausente ganha um rótulo: “Desconhecido”, “Sem Estado/Flair” ou “Post sem texto (Link ou Imagem)”.


### Carregamento dos dados  e ajustes de pivacidade dos csvs

In [47]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import hashlib
import os
import re
from dotenv import load_dotenv

PASTA = Path("../data")
PASTA_LIMPA = Path("../data_limpo")
PASTA_LIMPA.mkdir(
    parents=True, exist_ok=True
)


In [48]:
load_dotenv(dotenv_path="hash_salt.env")

SECRET_SALT = os.environ.get("HASH_SALT")

def anonimizar_com_hash(nome):
  if pd.isna(nome) or nome in ["Desconhecido", ""]:
    return nome
  
  if str(nome).startswith("User_"):
    return nome
  
  texto_combinado = SECRET_SALT + str(nome)
  hash_gerado = hashlib.md5(texto_combinado.encode("utf-8")).hexdigest()
  return f"User_{hash_gerado}"


def extrair_id_post(link):
  if pd.isna(link):
    return "Desconhecido"
  
  partes = str(link).split("/")

  if "comments" in partes:
    idx = partes.index("comments")
    if idx + 1 < len(partes):
      return partes[idx + 1]
  return "Desconhecido"


In [49]:
for caminho in PASTA.glob("*.csv"):
  df = pd.read_csv(caminho, encoding="utf-8")

  if "Link" in df.columns and "AutorPost" in df.columns:
    df["AutorPost"] = df["Link"].apply(extrair_id_post)

  cols_autor_com = [c for c in df.columns if c.startswith("AutorCom")]
  for col in cols_autor_com:
    if col in df.columns:
      df[col] = df[col].apply(anonimizar_com_hash)

  nome_saida = f"{caminho.stem}_limpo.csv"
  caminho_saida = PASTA_LIMPA / nome_saida
  df.to_csv(caminho_saida, index=False, encoding="utf-8")
  print(f"Criado em data_limpo: {nome_saida}")

Criado em data_limpo: actuallesbians_pastyear_limpo.csv
Criado em data_limpo: gayrelationship_pastyear_limpo.csv
Criado em data_limpo: interracial_relationships_pastyear_limpo.csv
Criado em data_limpo: relationship_advice_pastyear_limpo.csv
Criado em data_limpo: relationship_advice_limpo.csv
Criado em data_limpo: relationships_top_alltime_limpo.csv


## 1. r/relationships

In [50]:
nomes_desejados = ["relationship_advice_limpo.csv", "relationship_advice_pastyear_limpo.csv", "relationships_top_alltime_limpo.csv"]
arquivos = [PASTA_LIMPA / nome for nome in nomes_desejados]


In [51]:
for caminho in arquivos:
  df = pd.read_csv(caminho, encoding="utf-8")

  nulos_df = pd.DataFrame({
      "nulos": df.isna().sum(),
      "%": (df.isna().mean() * 100).round(2),
      "preenchidos": df.notna().sum(),
  })

  nulos_filtrado = nulos_df[nulos_df["nulos"] > 0].sort_values(
      "%", ascending=True
  )

  display(nulos_filtrado)

,nulos,%,preenchidos
AutorCom 4,1,0.21,486
Comentário 4,1,0.21,486
AutorCom 5,1,0.21,486
Comentário 5,1,0.21,486
AutorCom 6,1,0.21,486
...,...,...,...
Comentário 98,5,1.03,482
AutorCom 99,5,1.03,482
Comentário 99,5,1.03,482
AutorCom 100,5,1.03,482


,nulos,%,preenchidos
AutorCom 35,1,0.22,461
Comentário 35,1,0.22,461
AutorCom 36,1,0.22,461
Comentário 36,1,0.22,461
AutorCom 37,1,0.22,461
...,...,...,...
Comentário 98,27,5.84,435
AutorCom 99,29,6.28,433
Comentário 99,29,6.28,433
AutorCom 100,29,6.28,433


,nulos,%,preenchidos
AutorCom 1,2,0.41,489
Comentário 1,2,0.41,489
AutorCom 2,2,0.41,489
Comentário 2,2,0.41,489
AutorCom 3,2,0.41,489
...,...,...,...
Comentário 98,60,12.22,431
AutorCom 99,61,12.42,430
Comentário 99,61,12.42,430
AutorCom 100,63,12.83,428


In [52]:
for caminho in arquivos:
  df = pd.read_csv(caminho, encoding="utf-8")

  if "AutorCom 1" in df.columns:
    df = df[df["AutorCom 1"].notna()].copy()
    
  df.to_csv(caminho, index=False, encoding="utf-8")

## 2. r/interracialdating

In [53]:
nome_arquivo = "interracial_relationships_pastyear_limpo.csv"
caminho = PASTA_LIMPA / nome_arquivo

print(f"{'='*50}")
print(f"Analisando arquivo: {caminho.name}")
print(f"{'='*50}")

df = pd.read_csv(caminho, encoding="utf-8")

nulos_df = pd.DataFrame({
    "nulos": df.isna().sum(),
    "%": (df.isna().mean() * 100).round(2),
    "preenchidos": df.notna().sum(),
})

nulos_filtrado = nulos_df[nulos_df["nulos"] > 0].sort_values("%", ascending=True)

display(nulos_filtrado)


Analisando arquivo: interracial_relationships_pastyear_limpo.csv


,nulos,%,preenchidos
AutorCom 35,1,0.22,463
Comentário 35,1,0.22,463
AutorCom 36,1,0.22,463
Comentário 36,1,0.22,463
AutorCom 37,1,0.22,463
...,...,...,...
Comentário 98,28,6.03,436
AutorCom 99,30,6.47,434
Comentário 99,30,6.47,434
AutorCom 100,30,6.47,434


## 3. r/gayrelationships

In [54]:
nome_arquivo = "gayrelationship_pastyear_limpo.csv"
caminho = PASTA_LIMPA / nome_arquivo

print(f"{'='*50}")
print(f"Analisando arquivo: {caminho.name}")
print(f"{'='*50}")

df = pd.read_csv(caminho, encoding="utf-8")

nulos_df = pd.DataFrame({
    "nulos": df.isna().sum(),
    "%": (df.isna().mean() * 100).round(2),
    "preenchidos": df.notna().sum(),
})

nulos_filtrado = nulos_df[nulos_df["nulos"] > 0].sort_values("%", ascending=True)

display(nulos_filtrado)

Analisando arquivo: gayrelationship_pastyear_limpo.csv


,nulos,%,preenchidos
AutorCom 1,4,0.85,468
Comentário 1,4,0.85,468
AutorCom 2,14,2.97,458
Comentário 2,14,2.97,458
AutorCom 3,26,5.51,446
...,...,...,...
Comentário 98,472,100.00,0
AutorCom 99,472,100.00,0
Comentário 99,472,100.00,0
AutorCom 100,472,100.00,0


In [55]:

df = pd.read_csv(caminho, encoding="utf-8")

if "AutorCom 1" in df.columns:
    df = df[df["AutorCom 1"].notna()].copy()

df.to_csv(caminho, index=False, encoding="utf-8")


## 4. r/actuallesbians

In [56]:
nome_arquivo = "actuallesbians_pastyear_limpo.csv"
caminho = PASTA_LIMPA / nome_arquivo

print(f"{'='*50}")
print(f"Analisando arquivo: {caminho.name}")
print(f"{'='*50}")

df = pd.read_csv(caminho, encoding="utf-8")

nulos_df = pd.DataFrame({
    "nulos": df.isna().sum(),
    "%": (df.isna().mean() * 100).round(2),
    "preenchidos": df.notna().sum(),
})

nulos_filtrado = nulos_df[nulos_df["nulos"] > 0].sort_values("%", ascending=True)

display(nulos_filtrado)

Analisando arquivo: actuallesbians_pastyear_limpo.csv


,nulos,%,preenchidos
AutorCom 7,1,0.22,453
Comentário 7,1,0.22,453
AutorCom 8,4,0.88,450
Comentário 8,4,0.88,450
AutorCom 9,5,1.10,449
...,...,...,...
Comentário 98,332,73.13,122
AutorCom 99,333,73.35,121
Comentário 99,333,73.35,121
AutorCom 100,333,73.35,121
